# Chapter 3: Fused kernels and parallel patterns
This notebook implements the concepts and kernels from Chapter 3: analyzing arithmetic intensity, the memory wall, fusing kernels (SiLU, SwiGLU), and advanced coordinate mapping with Rotary Positional Embeddings (RoPE).

## Embarrassingly parallel tasks: the baseline GPU pattern
Element-wise operations are entirely independent, allowing straightforward parallelization across blocks.
Below is the complete Triton implementation of the Sigmoid Linear Unit (SiLU) activation function.

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def silu_kernel(input_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    
    x = tl.load(input_ptr + offsets, mask=mask)
    y = tl.sigmoid(x) * x
    tl.store(output_ptr + offsets, y, mask=mask)

def solve(input_tensor: torch.Tensor, output_tensor: torch.Tensor, N: int):
    BLOCK_SIZE = 1024
    grid = (triton.cdiv(N, BLOCK_SIZE),)
    silu_kernel[grid](input_tensor, output_tensor, N, BLOCK_SIZE)

N = 1024
input_tensor = torch.randn(N, device='cuda')
output_tensor = torch.empty_like(input_tensor)
solve(input_tensor, output_tensor, N)
print("SiLU kernel executed successfully.")

## The problem with unfused operations
Chaining multiple element-wise operations together in PyTorch causes intermediate results to travel through global memory multiple times. Below is an example of an unfused chain.

In [ ]:
import torch.nn.functional as F

x = torch.randn(10, device='cuda')
y = torch.randn(10, device='cuda')
silu = F.silu

temp = silu(x)
output = temp * y

print("Unfused sequence executed.")

## Case Study: SwiGLU
In a naive SwiGLU implementation, Swish logic and multiplicative gating require intermediate materialization matrices, decreasing computational intensity.

In [ ]:
x_swi = torch.randn(1, 1024, device='cuda')
W = torch.randn(1024, 1024, device='cuda')
V = torch.randn(1024, 1024, device='cuda')

gate = silu(x_swi @ W)           
up = x_swi @ V                   
output = gate * up 

print("Naive SwiGLU executed.")

### Fused SwiGLU kernel
By fusing steps across registers cleanly, we avoid intermediate trips to global memory. The kernel computes the element-wise operations immediately following a matrix multiplication.

In [ ]:
@triton.jit
def swiglu(input_ptr, output_ptr, N, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0) 

    H = N // 2 
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE) 
    mask = offs < H

    x1 = tl.load(input_ptr + offs, mask=mask, other=0.0) 
    x2 = tl.load(input_ptr + H + offs, mask=mask, other=0.0) 

    silu_val = x1 * tl.sigmoid(x1) 
    out_val = silu_val * x2 

    tl.store(output_ptr + offs, out_val, mask=mask) 

N_swiglu = 1024
H_swiglu = N_swiglu // 2
input_fused = torch.randn(N_swiglu, device='cuda')
output_fused = torch.empty(H_swiglu, device='cuda')

grid_swiglu = (triton.cdiv(H_swiglu, 1024),)
swiglu[grid_swiglu](input_fused, output_fused, N_swiglu, BLOCK_SIZE=1024)

print("Fused SwiGLU kernel executed successfully.")

## Control flow: masking over branching
Using traditional branching logic like if/else (demonstrated below with ReLU) introduces warp divergence within the SIMT model. Triton resolves this via predication, avoiding serialization penalties.

In [ ]:
input_val = -1.0

if input_val < 0:
    output_val = 0
else:
    output_val = input_val
    
print(f"ReLU branching output: {output_val}")

## Advanced indexing: coordinate mapping (RoPE)
The translation of a logical position to physical memory addresses. Rotary Positional Embeddings require loading multi-dimensional blocks using disparate stride coordinates.

In [ ]:
@triton.jit
def rope_kernel(
    x_ptr,
    output_ptr,
    cos_ptr,
    sin_ptr,
    seq_len,
    feature_dim,
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(0) 
    half_dim = feature_dim // 2 
    local_idx = tl.arange(0, BLOCK_SIZE)
    mask = local_idx < half_dim
    row_start = pid * feature_dim 
    first_offset = row_start + local_idx
    second_offset = row_start + local_idx + half_dim 

    x1 = tl.load(x_ptr + first_offset, mask=mask, other=0.0)
    x2 = tl.load(x_ptr + second_offset, mask=mask, other=0.0)

    angle_row_start = pid * half_dim 
    angle_offset = angle_row_start + local_idx

    cos_val = tl.load(cos_ptr + angle_offset, mask=mask, other=0.0) 
    sin_val = tl.load(sin_ptr + angle_offset, mask=mask, other=0.0)

    out1 = x1 * cos_val - x2 * sin_val 
    out2 = x2 * cos_val + x1 * sin_val

    tl.store(output_ptr + first_offset, out1, mask=mask)
    tl.store(output_ptr + second_offset, out2, mask=mask)

seq_length = 2
f_dim = 128
blocks = 64

x_rope = torch.randn(seq_length * f_dim, device='cuda')
out_rope = torch.empty_like(x_rope)
cos_grid = torch.randn(seq_length * (f_dim // 2), device='cuda')
sin_grid = torch.randn(seq_length * (f_dim // 2), device='cuda')

grid_rope = (seq_length,)
rope_kernel[grid_rope](x_rope, out_rope, cos_grid, sin_grid, seq_length, f_dim, BLOCK_SIZE=blocks)

print("RoPE kernel executed successfully.")